In [ ]:

import os
import sys
import torch
import matplotlib

IN_COLAB    = 'google.colab' in sys.modules
IN_KAGGLE   = os.path.exists('/kaggle/working')
IN_NOTEBOOK = 'ipykernel' in sys.modules

# Use non-interactive backend when running headless scripts (prevents blocking)
if not IN_NOTEBOOK:
    matplotlib.use('Agg')

if IN_COLAB:
    from google.colab import drive, output
    drive.mount('/content/drive')
    output.enable_custom_widget_manager()
    PROJECT_ROOT = '/content/drive/MyDrive/PINN_ImpactDamage'
    ENV_NAME = 'Google Colab'
elif IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/PINN_ImpactDamage'
    ENV_NAME = 'Kaggle'
else:
    PROJECT_ROOT = os.path.abspath('./outputs')
    ENV_NAME = 'Local Machine'

FOLDERS = [
    PROJECT_ROOT,
    f'{PROJECT_ROOT}/01_checkpoints/elasticity',
    f'{PROJECT_ROOT}/01_checkpoints/damage',
    f'{PROJECT_ROOT}/01_checkpoints/debris',
    f'{PROJECT_ROOT}/02_results/stress_maps',
    f'{PROJECT_ROOT}/02_results/damage_maps',
    f'{PROJECT_ROOT}/02_results/debris_plots',
    f'{PROJECT_ROOT}/02_results/dashboards',
    f'{PROJECT_ROOT}/03_logs/elasticity',
    f'{PROJECT_ROOT}/03_logs/damage',
    f'{PROJECT_ROOT}/03_logs/debris',
    f'{PROJECT_ROOT}/04_exports',
    f'{PROJECT_ROOT}/05_configs',
    f'{PROJECT_ROOT}/06_data',
]
for _folder in FOLDERS:
    os.makedirs(_folder, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device('cuda')
    GPU_NAME = torch.cuda.get_device_name(0)
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    GPU_NAME = 'Apple Silicon MPS'
else:
    device = torch.device('cpu')
    GPU_NAME = 'CPU only'


def safe_torch_load(path, map_location=device):
    """Compatible with PyTorch <2.6 and >=2.6 (weights_only default change)."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


CONFIG = {
    # ── Paths ──────────────────────────────────────────────
    'root'          : PROJECT_ROOT,
    'ckpt_elastic'  : f'{PROJECT_ROOT}/01_checkpoints/elasticity',
    'ckpt_damage'   : f'{PROJECT_ROOT}/01_checkpoints/damage',
    'ckpt_debris'   : f'{PROJECT_ROOT}/01_checkpoints/debris',
    'res_stress'    : f'{PROJECT_ROOT}/02_results/stress_maps',
    'res_damage'    : f'{PROJECT_ROOT}/02_results/damage_maps',
    'res_debris'    : f'{PROJECT_ROOT}/02_results/debris_plots',
    'res_dashboard' : f'{PROJECT_ROOT}/02_results/dashboards',
    'logs'          : f'{PROJECT_ROOT}/03_logs',
    'exports'       : f'{PROJECT_ROOT}/04_exports',
    'configs'       : f'{PROJECT_ROOT}/05_configs',

    # ── Material: Structural Steel ──────────────────────────
    'E'             : 200e9,    # Young's modulus (Pa)
    'nu'            : 0.30,     # Poisson's ratio
    'rho'           : 7850.0,   # Density (kg/m^3)
    'sigma_y'       : 250e6,    # Yield stress (Pa)
    'sigma_u'       : 400e6,    # Ultimate tensile strength (Pa)
    'G_c'           : 50e3,     # Fracture energy (J/m^2)
    'damage_thresh' : 0.85,     # D > threshold => fragment breaks off

    # ── PINN Training Defaults ─────────────────────────────
    'N_col'         : 5000,
    'N_bc'          : 300,
    'N_ic'          : 300,
    'epochs_elast'  : 2000,
    'epochs_damage' : 2000,
    'epochs_debris' : 1500,
    'lbfgs_steps'   : 300,      # L-BFGS polishing iterations after Adam
    'resample_every': 50,       # Resample interior LHS points every N Adam epochs
    'lr'            : 1e-3,
    'lr_decay_step' : 1000,
    'lr_decay_rate' : 0.5,

    # ── Network Architecture ────────────────────────────────
    'layers_elast'  : [4, 128, 128, 128, 128, 128, 3],
    'layers_damage' : [4, 128, 128, 128, 128, 1],
    'layers_debris' : [1, 64, 64, 64, 3],

    # ── Loss Weights ────────────────────────────────────────
    'w_pde'         : 1.0,
    'w_bc'          : 10.0,
    'w_ic'          : 10.0,

    # ── Simulation ──────────────────────────────────────────
    't_end'         : 1.0,
    'n_fragments'   : 5,
}

print(f"✅ Environment  : {ENV_NAME}")
print(f"✅ Device       : {device} ({GPU_NAME})")
print(f"✅ PyTorch      : {torch.__version__}")
print(f"✅ Output root  : {PROJECT_ROOT}")


In [ ]:
# ============================================================
# PHYSICS PARAMETER GENERATOR & LATIN HYPERCUBE SAMPLER
# ============================================================

import json
import numpy as np
from scipy.stats import qmc


def lhs_sample(n_samples, n_dims, seed=None):
    """Generate Latin Hypercube samples in [0, 1]^n_dims."""
    sampler = qmc.LatinHypercube(d=n_dims, seed=seed)
    return sampler.random(n=n_samples)


def compute_physics_params(params):
    """Convert raw user inputs into normalized physics quantities."""
    m              = params['mass']
    v              = params['velocity']
    d              = params['diameter']
    ang            = params['angle_deg']
    L              = params['length']
    W              = params['width']
    H              = params['thickness']
    x0             = params['impact_x']
    y0             = params['impact_y']
    plate_height_m = params['plate_height_m']

    print("=" * 55)
    print("  PHYSICS PARAMETER GENERATOR")
    print("=" * 55)
    print(f"\n📥 Raw Inputs:")
    print(f"   Projectile  : mass={m} kg | velocity={v} m/s | d={d} m | angle={ang}°")
    print(f"   Plate       : {L}m x {W}m x {H}m | height={plate_height_m} m")
    print(f"   Impact point: ({x0}, {y0}) normalized")

    angle_rad  = np.radians(ang)
    t_contact  = d / v
    F_impact   = (m * v * np.sin(angle_rad)) / t_contact
    F_impact_n = F_impact / (CONFIG['E'] * L * W)

    KE              = 0.5 * m * v**2
    contact_area    = np.pi * (d / 2.0)**2
    impact_pressure = F_impact / contact_area
    pressure_norm   = impact_pressure / CONFIG['E']

    r_x   = (d / 2.0) / L
    r_y   = (d / 2.0) / W
    r_avg = max((r_x + r_y) / 2.0, 0.02)

    E, nu, rho = CONFIG['E'], CONFIG['nu'], CONFIG['rho']
    c_p      = np.sqrt(E * (1.0 - nu) / (rho * (1.0 + nu) * (1.0 - 2.0 * nu)))
    c_s      = np.sqrt(E / (2.0 * rho * (1.0 + nu)))
    t_cross  = L / c_p
    t_peak_n = min(t_contact / t_cross, 0.15)

    z_launch_abs  = plate_height_m + H
    z_launch_norm = z_launch_abs / L

    boundary_conditions = {
        'x_min': 'clamped', 'x_max': 'clamped',
        'y_min': 'clamped', 'y_max': 'clamped',
        'z_min': 'free',    'z_top': 'free',
    }

    sigma_y_norm  = CONFIG['sigma_y'] / E
    debris_v_init = v * 0.15
    debris_v_norm = debris_v_init / (L / t_cross)

    print(f"\n⚡ Impact & Wave Scaling:")
    print(f"   Contact time    : {t_contact*1e6:.3f} us | F_impact: {F_impact/1e6:.4f} MN")
    print(f"   Kinetic energy  : {KE/1e3:.2f} kJ | Pressure: {impact_pressure/1e6:.2f} MPa")
    print(f"   c_p={c_p:.1f} m/s | c_s={c_s:.1f} m/s | t_cross={t_cross*1e6:.2f} us")
    print(f"   Debris launch Z : {z_launch_abs:.3f} m ({z_launch_norm:.3f} norm)")

    physics = {
        'mass'            : float(m),
        'velocity'        : float(v),
        'diameter'        : float(d),
        'angle_deg'       : float(ang),
        'L'               : float(L),
        'W'               : float(W),
        'H'               : float(H),
        'x0'              : float(x0),
        'y0'              : float(y0),
        'z0'              : 1.0,
        'z_launch_abs'    : float(z_launch_abs),
        'z_launch_norm'   : float(z_launch_norm),
        'plate_height_m'  : float(plate_height_m),
        'F_impact'        : float(F_impact),
        'F_norm'          : float(F_impact_n),
        'impact_pressure' : float(impact_pressure),
        'pressure_norm'   : float(pressure_norm),
        'KE'              : float(KE),
        'r_x'             : float(r_x),
        'r_y'             : float(r_y),
        'r_avg'           : float(r_avg),
        'c_p'             : float(c_p),
        'c_s'             : float(c_s),
        't_cross'         : float(t_cross),
        't_contact'       : float(t_contact),
        't_peak_n'        : float(t_peak_n),
        't_sigma'         : 0.02,
        'lam_n'           : float((E * nu) / ((1.0 + nu) * (1.0 - 2.0 * nu)) / E),
        'mu_n'            : float(E / (2.0 * (1.0 + nu)) / E),
        'rho_n'           : 1.0,
        'sigma_y_norm'    : float(sigma_y_norm),
        'BC'              : boundary_conditions,
        'debris_v_norm'   : float(debris_v_norm),
        'n_fragments'     : CONFIG['n_fragments'],
        'epochs'          : int(params['epochs']),
        'n_col'           : int(params['n_col']),
        'lbfgs_steps'     : int(params.get('lbfgs_steps', CONFIG['lbfgs_steps'])),
        'resample_every'  : int(params.get('resample_every', CONFIG['resample_every'])),
        'resume'          : bool(params.get('resume', False)),
    }

    ts = params['timestamp']
    fname = f"{CONFIG['configs']}/physics_{ts}.json"
    with open(fname, 'w') as f:
        json.dump(physics, f, indent=2)

    print(f"✅ Physics params saved -> {fname}\n")
    return physics


In [ ]:
# ============================================================
# 3D ELASTICITY PINN (Same Navier Physics + Batched BCs + Adam & L-BFGS)
# ============================================================

import os
import json
import torch
import torch.nn as nn
from tqdm.auto import tqdm


class ElasticityPINN(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.depth      = len(layers) - 1
        self.activation = nn.Tanh()
        self.linears    = nn.ModuleList([
            nn.Linear(layers[i], layers[i + 1]) for i in range(self.depth)
        ])
        for layer in self.linears:
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x, y, z, t):
        out = torch.cat([x, y, z, t], dim=1)
        for i in range(self.depth - 1):
            out = self.activation(self.linears[i](out))
        out = self.linears[-1](out)
        return out[:, 0:1], out[:, 1:2], out[:, 2:3]


def make_grad_tensor(data, dev=device):
    return torch.tensor(data, dtype=torch.float32, device=dev).requires_grad_(True)


def rand_grad(shape, dev=device, val=None):
    if val is not None:
        t = torch.full(shape, val, dtype=torch.float32, device=dev)
    else:
        t = torch.rand(shape, dtype=torch.float32, device=dev)
    return t.requires_grad_(True)


def impact_force(x, y, z, t, physics):
    x0, y0  = physics['x0'], physics['y0']
    r       = physics['r_avg']
    t_peak  = physics['t_peak_n']
    t_sigma = physics['t_sigma']
    F       = physics['pressure_norm']

    spatial  = torch.exp(-((x - x0)**2 + (y - y0)**2) / (2.0 * r**2))
    temporal = torch.exp(-((t - t_peak)**2) / (t_sigma**2))
    z_mask   = torch.exp(-((z - 1.0)**2) / (0.05**2))

    fx = torch.zeros_like(x)
    fy = torch.zeros_like(y)
    fz = -F * spatial * temporal * z_mask
    return fx, fy, fz


def compute_stress(model, x, y, z, t):
    u, v, w = model(x, y, z, t)

    def grad(f, var):
        return torch.autograd.grad(
            f, var, grad_outputs=torch.ones_like(f),
            create_graph=True, allow_unused=False
        )[0]

    u_x, u_y, u_z = grad(u, x), grad(u, y), grad(u, z)
    v_x, v_y, v_z = grad(v, x), grad(v, y), grad(v, z)
    w_x, w_y, w_z = grad(w, x), grad(w, y), grad(w, z)

    eps_xx, eps_yy, eps_zz = u_x, v_y, w_z
    eps_xy = 0.5 * (u_y + v_x)
    eps_xz = 0.5 * (u_z + w_x)
    eps_yz = 0.5 * (v_z + w_y)

    nu    = CONFIG['nu']
    lam_n = nu / ((1.0 + nu) * (1.0 - 2.0 * nu))
    mu_n  = 1.0 / (2.0 * (1.0 + nu))
    trace = eps_xx + eps_yy + eps_zz

    sig_xx = lam_n * trace + 2.0 * mu_n * eps_xx
    sig_yy = lam_n * trace + 2.0 * mu_n * eps_yy
    sig_zz = lam_n * trace + 2.0 * mu_n * eps_zz
    sig_xy = 2.0 * mu_n * eps_xy
    sig_xz = 2.0 * mu_n * eps_xz
    sig_yz = 2.0 * mu_n * eps_yz

    return (sig_xx, sig_yy, sig_zz, sig_xy, sig_xz, sig_yz,
            u, v, w, u_x, v_y, w_z, u_y, u_z, v_x, v_z, w_x, w_y)


def loss_pde(model, x, y, z, t, physics):
    (sig_xx, sig_yy, sig_zz,
     sig_xy, sig_xz, sig_yz,
     u, v, w, *_) = compute_stress(model, x, y, z, t)

    def grad(f, var):
        return torch.autograd.grad(
            f, var, grad_outputs=torch.ones_like(f),
            create_graph=True, allow_unused=False
        )[0]

    u_tt = grad(grad(u, t), t)
    v_tt = grad(grad(v, t), t)
    w_tt = grad(grad(w, t), t)

    fx, fy, fz = impact_force(x, y, z, t, physics)

    res_x = grad(sig_xx, x) + grad(sig_xy, y) + grad(sig_xz, z) + fx - u_tt
    res_y = grad(sig_xy, x) + grad(sig_yy, y) + grad(sig_yz, z) + fy - v_tt
    res_z = grad(sig_xz, x) + grad(sig_yz, y) + grad(sig_zz, z) + fz - w_tt

    return torch.mean(res_x**2) + torch.mean(res_y**2) + torch.mean(res_z**2)


def sample_bc_points(N=300):
    zeros = torch.zeros((N, 1), device=device)
    ones  = torch.ones((N, 1), device=device)
    r = lambda: torch.rand((N, 1), device=device)

    xe = torch.cat([zeros, ones, r(),   r()],  dim=0)
    ye = torch.cat([r(),   r(),  zeros, ones], dim=0)
    ze = torch.rand((4 * N, 1), device=device)
    te = torch.rand((4 * N, 1), device=device)

    xf = torch.rand((2 * N, 1), device=device).requires_grad_(True)
    yf = torch.rand((2 * N, 1), device=device).requires_grad_(True)
    zf = torch.cat([zeros, ones], dim=0).requires_grad_(True)
    tf = torch.rand((2 * N, 1), device=device).requires_grad_(True)

    return (xe, ye, ze, te), (xf, yf, zf, tf)


def loss_bc(model, physics, N=300, bc_pts=None):
    if bc_pts is None:
        bc_pts = sample_bc_points(N)
    (xe, ye, ze, te), (xf, yf, zf, tf) = bc_pts

    ub, vb, wb = model(xe, ye, ze, te)
    loss_edges = 4.0 * (torch.mean(ub**2) + torch.mean(vb**2) + torch.mean(wb**2))

    (_, _, sig_zz, _, sig_xz, sig_yz, *_) = compute_stress(model, xf, yf, zf, tf)
    loss_faces = 2.0 * (torch.mean(sig_zz**2) + torch.mean(sig_xz**2) + torch.mean(sig_yz**2))

    return loss_edges + loss_faces


def sample_ic_points(N=300):
    xi = rand_grad((N, 1), device)
    yi = rand_grad((N, 1), device)
    zi = rand_grad((N, 1), device)
    ti = rand_grad((N, 1), device, val=0.0)
    return xi, yi, zi, ti


def loss_ic(model, N=300, ic_pts=None):
    xi, yi, zi, ti = sample_ic_points(N) if ic_pts is None else ic_pts
    u, v, w = model(xi, yi, zi, ti)
    loss_disp = torch.mean(u**2) + torch.mean(v**2) + torch.mean(w**2)

    def grad(f, var):
        return torch.autograd.grad(
            f, var, grad_outputs=torch.ones_like(f), create_graph=True
        )[0]

    loss_vel = (torch.mean(grad(u, ti)**2) +
                torch.mean(grad(v, ti)**2) +
                torch.mean(grad(w, ti)**2))
    return loss_disp + loss_vel


def sample_collocation(n, physics=None, seed=None):
    samples = lhs_sample(n, 4, seed=seed)
    x = make_grad_tensor(samples[:, 0:1], device)
    y = make_grad_tensor(samples[:, 1:2], device)
    z = make_grad_tensor(samples[:, 2:3], device)
    t = make_grad_tensor(samples[:, 3:4], device)
    return x, y, z, t


def train_elasticity_pinn(physics, run_tag='default'):
    print("\n" + "=" * 55)
    print("  ELASTICITY PINN — TRAINING (Adam + L-BFGS)")
    print("=" * 55)

    epochs         = physics['epochs']
    n_col          = physics['n_col']
    resume         = physics['resume']
    lbfgs_steps    = physics.get('lbfgs_steps', CONFIG['lbfgs_steps'])
    resample_every = physics.get('resample_every', CONFIG['resample_every'])

    ckpt_path = f"{CONFIG['ckpt_elastic']}/elasticity_{run_tag}.pth"
    log_path  = f"{CONFIG['logs']}/elasticity/loss_{run_tag}.json"

    model = ElasticityPINN(CONFIG['layers_elast']).to(device)
    start_epoch = 0

    if resume and os.path.exists(ckpt_path):
        ckpt = safe_torch_load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        start_epoch = ckpt['epoch']
        print(f"✅ Resumed from epoch {start_epoch}")

    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=CONFIG['lr_decay_step'], gamma=CONFIG['lr_decay_rate']
    )

    x_col, y_col, z_col, t_col = sample_collocation(n_col, physics)
    history   = {'total': [], 'pde': [], 'bc': [], 'ic': [], 'epochs': []}
    best_loss = float('inf')
    save_every = max(epochs // 10, 500)

    # ── Stage 1: Adam with periodic LHS resampling ────────
    for epoch in tqdm(range(start_epoch, start_epoch + epochs), desc="Elasticity (Adam)"):
        if epoch > start_epoch and (epoch - start_epoch) % resample_every == 0:
            x_col, y_col, z_col, t_col = sample_collocation(n_col, physics)

        optimizer.zero_grad()
        l_pde = loss_pde(model, x_col, y_col, z_col, t_col, physics)
        l_bc  = loss_bc(model, physics, N=CONFIG['N_bc'])
        l_ic  = loss_ic(model, N=CONFIG['N_ic'])
        loss  = CONFIG['w_pde'] * l_pde + CONFIG['w_bc'] * l_bc + CONFIG['w_ic'] * l_ic

        loss.backward()
        optimizer.step()
        scheduler.step()

        history['total'].append(loss.item())
        history['pde'].append(l_pde.item())
        history['bc'].append(l_bc.item())
        history['ic'].append(l_ic.item())
        history['epochs'].append(epoch)

        if epoch % 500 == 0:
            tqdm.write(f"Epoch {epoch:6d} | Total: {loss.item():.5f} | "
                       f"PDE: {l_pde.item():.5f} | BC: {l_bc.item():.5f} | IC: {l_ic.item():.5f}")

        if loss.item() < best_loss or (epoch % save_every == 0 and epoch > start_epoch):
            best_loss = min(loss.item(), best_loss)
            torch.save({
                'epoch': epoch, 'model_state': model.state_dict(),
                'loss': loss.item(),
                'physics': {k: v for k, v in physics.items() if k != 'BC'},
            }, ckpt_path)

    # ── Stage 2: L-BFGS Polishing ─────────────────────────
    if lbfgs_steps > 0:
        print(f"🔧 Polishing Elasticity PINN with L-BFGS (max {lbfgs_steps} iters)...")
        x_col, y_col, z_col, t_col = sample_collocation(n_col, physics)
        bc_fixed = sample_bc_points(N=400)
        ic_fixed = sample_ic_points(N=400)

        lbfgs = torch.optim.LBFGS(
            model.parameters(), lr=1.0, max_iter=lbfgs_steps,
            history_size=50, line_search_fn='strong_wolfe'
        )
        step_counter = [start_epoch + epochs]

        def closure():
            lbfgs.zero_grad()
            l_pde = loss_pde(model, x_col, y_col, z_col, t_col, physics)
            l_bc  = loss_bc(model, physics, bc_pts=bc_fixed)
            l_ic  = loss_ic(model, ic_pts=ic_fixed)
            loss  = CONFIG['w_pde'] * l_pde + CONFIG['w_bc'] * l_bc + CONFIG['w_ic'] * l_ic
            loss.backward()

            history['total'].append(loss.item())
            history['pde'].append(l_pde.item())
            history['bc'].append(l_bc.item())
            history['ic'].append(l_ic.item())
            history['epochs'].append(step_counter[0])
            step_counter[0] += 1
            return loss

        lbfgs.step(closure)

    torch.save({
        'epoch': history['epochs'][-1], 'model_state': model.state_dict(),
        'loss': history['total'][-1],
        'physics': {k: v for k, v in physics.items() if k != 'BC'},
    }, ckpt_path)

    with open(log_path, 'w') as f:
        json.dump(history, f)

    print(f"✅ Elasticity PINN complete | Final loss: {history['total'][-1]:.6f}")
    return model, history


In [ ]:
# ============================================================
# DAMAGE PINN (Cached Von Mises Stress + O(1) Scaling + Adam & L-BFGS)
# ============================================================

import os
import json
import torch
import torch.nn as nn
from tqdm.auto import tqdm


class DamagePINN(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.depth      = len(layers) - 1
        self.activation = nn.Tanh()
        self.linears    = nn.ModuleList([
            nn.Linear(layers[i], layers[i + 1]) for i in range(self.depth)
        ])
        for layer in self.linears:
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x, y, z, t):
        out = torch.cat([x, y, z, t], dim=1)
        for i in range(self.depth - 1):
            out = self.activation(self.linears[i](out))
        return torch.sigmoid(self.linears[-1](out))


def von_mises_stress(model_e, x, y, z, t):
    (sig_xx, sig_yy, sig_zz, sig_xy, sig_xz, sig_yz, *_) = compute_stress(model_e, x, y, z, t)
    s11, s22, s33 = sig_xx.detach(), sig_yy.detach(), sig_zz.detach()
    s12, s13, s23 = sig_xy.detach(), sig_xz.detach(), sig_yz.detach()
    return torch.sqrt(0.5 * (
        (s11 - s22)**2 + (s22 - s33)**2 + (s33 - s11)**2 +
        6.0 * (s12**2 + s13**2 + s23**2)
    ) + 1e-8)


def loss_damage_pde(model_d, x, y, z, t, vm_cached, physics):
    """
    Dimensionless CDM evolution PDE:
      dD/dt = H(vm - sigma_y) * ((vm - sigma_y) / sigma_y)^n
    Uses cached vm_cached so model_e is not re-differentiated every epoch.
    """
    sigma_y = physics['sigma_y_norm']
    n_exp   = 2.0
    k       = 1000.0

    D   = model_d(x, y, z, t)
    D_t = torch.autograd.grad(
        D, t, grad_outputs=torch.ones_like(D), create_graph=True
    )[0]

    heaviside     = torch.sigmoid(k * (vm_cached - sigma_y))
    excess_stress = torch.clamp((vm_cached - sigma_y) / sigma_y, min=0.0)
    driving       = heaviside * (excess_stress ** n_exp)

    irrev    = torch.clamp(-D_t, min=0.0)**2
    residual = D_t - driving

    return torch.mean(residual**2) + 10.0 * torch.mean(irrev)


def sample_damage_bc(N=300):
    zeros = torch.zeros((N, 1), device=device)
    ones  = torch.ones((N, 1), device=device)
    r = lambda: torch.rand((N, 1), device=device)
    xb = torch.cat([zeros, ones, r(),   r()],  dim=0)
    yb = torch.cat([r(),   r(),  zeros, ones], dim=0)
    zb = torch.rand((4 * N, 1), device=device)
    tb = torch.rand((4 * N, 1), device=device)
    return xb, yb, zb, tb


def loss_damage_bc(model_d, N=300, bc_pts=None):
    xb, yb, zb, tb = sample_damage_bc(N) if bc_pts is None else bc_pts
    return 4.0 * torch.mean(model_d(xb, yb, zb, tb)**2)


def sample_damage_ic(N=300):
    return (torch.rand((N, 1), device=device),
            torch.rand((N, 1), device=device),
            torch.rand((N, 1), device=device),
            torch.zeros((N, 1), device=device))


def loss_damage_ic(model_d, N=300, ic_pts=None):
    xi, yi, zi, ti = sample_damage_ic(N) if ic_pts is None else ic_pts
    return torch.mean(model_d(xi, yi, zi, ti)**2)


def train_damage_pinn(model_e, physics, run_tag='default'):
    print("\n" + "=" * 55)
    print("  DAMAGE PINN — TRAINING (Cached Stress + Adam + L-BFGS)")
    print("=" * 55)

    epochs         = physics['epochs']
    n_col          = physics['n_col']
    resume         = physics['resume']
    lbfgs_steps    = physics.get('lbfgs_steps', CONFIG['lbfgs_steps'])
    resample_every = physics.get('resample_every', CONFIG['resample_every'])

    ckpt_path = f"{CONFIG['ckpt_damage']}/damage_{run_tag}.pth"
    log_path  = f"{CONFIG['logs']}/damage/loss_{run_tag}.json"

    model_e.eval()
    for p in model_e.parameters():
        p.requires_grad_(False)

    model_d = DamagePINN(CONFIG['layers_damage']).to(device)
    start_epoch = 0

    if resume and os.path.exists(ckpt_path):
        ckpt = safe_torch_load(ckpt_path, map_location=device)
        model_d.load_state_dict(ckpt['model_state'])
        start_epoch = ckpt['epoch']
        print(f"✅ Resumed from epoch {start_epoch}")

    optimizer = torch.optim.Adam(model_d.parameters(), lr=CONFIG['lr'])
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=CONFIG['lr_decay_step'], gamma=CONFIG['lr_decay_rate']
    )

    def resample_with_stress():
        xc, yc, zc, tc = sample_collocation(n_col, physics)
        vm = von_mises_stress(model_e, xc, yc, zc, tc)
        return xc.detach(), yc.detach(), zc.detach(), tc.detach().requires_grad_(True), vm

    x_col, y_col, z_col, t_col, vm_col = resample_with_stress()

    history   = {'total': [], 'pde': [], 'bc': [], 'ic': [], 'bounds': [], 'epochs': []}
    best_loss = float('inf')
    save_every = max(epochs // 10, 500)

    # ── Stage 1: Adam ─────────────────────────────────────
    for epoch in tqdm(range(start_epoch, start_epoch + epochs), desc="Damage (Adam)"):
        if epoch > start_epoch and (epoch - start_epoch) % resample_every == 0:
            x_col, y_col, z_col, t_col, vm_col = resample_with_stress()

        optimizer.zero_grad()
        l_pde = loss_damage_pde(model_d, x_col, y_col, z_col, t_col, vm_col, physics)
        l_bc  = loss_damage_bc(model_d, N=CONFIG['N_bc'])
        l_ic  = loss_damage_ic(model_d, N=CONFIG['N_ic'])
        loss  = CONFIG['w_pde'] * l_pde + CONFIG['w_bc'] * l_bc + CONFIG['w_ic'] * l_ic

        loss.backward()
        optimizer.step()
        scheduler.step()

        history['total'].append(loss.item())
        history['pde'].append(l_pde.item())
        history['bc'].append(l_bc.item())
        history['ic'].append(l_ic.item())
        history['bounds'].append(0.0)
        history['epochs'].append(epoch)

        if epoch % 500 == 0:
            tqdm.write(f"Epoch {epoch:6d} | Total: {loss.item():.5f} | "
                       f"PDE: {l_pde.item():.5f} | BC: {l_bc.item():.5f} | IC: {l_ic.item():.5f}")

        if loss.item() < best_loss or (epoch % save_every == 0 and epoch > start_epoch):
            best_loss = min(loss.item(), best_loss)
            torch.save({'epoch': epoch, 'model_state': model_d.state_dict(),
                        'loss': loss.item()}, ckpt_path)

    # ── Stage 2: L-BFGS Polishing ─────────────────────────
    if lbfgs_steps > 0:
        print(f"🔧 Polishing Damage PINN with L-BFGS (max {lbfgs_steps} iters)...")
        x_col, y_col, z_col, t_col, vm_col = resample_with_stress()
        bc_fixed = sample_damage_bc(N=400)
        ic_fixed = sample_damage_ic(N=400)

        lbfgs = torch.optim.LBFGS(
            model_d.parameters(), lr=1.0, max_iter=lbfgs_steps,
            history_size=50, line_search_fn='strong_wolfe'
        )
        step_counter = [start_epoch + epochs]

        def closure():
            lbfgs.zero_grad()
            l_pde = loss_damage_pde(model_d, x_col, y_col, z_col, t_col, vm_col, physics)
            l_bc  = loss_damage_bc(model_d, bc_pts=bc_fixed)
            l_ic  = loss_damage_ic(model_d, ic_pts=ic_fixed)
            loss  = CONFIG['w_pde'] * l_pde + CONFIG['w_bc'] * l_bc + CONFIG['w_ic'] * l_ic
            loss.backward()

            history['total'].append(loss.item())
            history['pde'].append(l_pde.item())
            history['bc'].append(l_bc.item())
            history['ic'].append(l_ic.item())
            history['bounds'].append(0.0)
            history['epochs'].append(step_counter[0])
            step_counter[0] += 1
            return loss

        lbfgs.step(closure)

    torch.save({'epoch': history['epochs'][-1], 'model_state': model_d.state_dict(),
                'loss': history['total'][-1]}, ckpt_path)
    with open(log_path, 'w') as f:
        json.dump(history, f)

    print(f"✅ Damage PINN complete | Final loss: {history['total'][-1]:.6f}")
    return model_d, history


In [ ]:
# ============================================================
# DEBRIS PINN (Adam + L-BFGS Polishing + scipy.solve_ivp Validation)
# ============================================================

import os
import json
import numpy as np
import torch
import torch.nn as nn
from scipy.integrate import solve_ivp
from tqdm.auto import tqdm


def get_debris_constants(physics, m_frag=0.01):
    """Shared normalized gravity and aerodynamic drag constants for PINN & ODE."""
    L       = physics['L']
    t_cross = physics['t_cross']
    g_norm  = 9.81 * (t_cross**2) / L
    Cd      = 0.47
    rho_air = 1.225
    r_frag  = 0.005
    A_frag  = np.pi * (r_frag**2)
    k_drag  = (0.5 * Cd * rho_air * A_frag * (t_cross**2)) / (m_frag * L)
    return g_norm, k_drag


def find_fracture_zones(model_d, physics, n_fragments=5):
    print("🔍 Checking fracture condition...\n")
    sigma_y  = CONFIG['sigma_y']
    sigma_u  = CONFIG['sigma_u']
    impact_p = physics['impact_pressure']

    print(f"   Impact pressure : {impact_p/1e6:.2f} MPa")
    print(f"   Yield stress    : {sigma_y/1e6:.2f} MPa")
    print(f"   Ultimate stress : {sigma_u/1e6:.2f} MPa")

    if impact_p < sigma_y:
        print("\n✅ ELASTIC REGIME — No fracture, no fragments formed")
        return []
    elif impact_p < sigma_u:
        damage_ratio = min((impact_p - sigma_y) / (sigma_u - sigma_y), 1.0)
        print(f"\n⚠️  PLASTIC REGIME | Damage ratio: {damage_ratio:.3f}")
        if damage_ratio < 0.5:
            print("   Damage ratio < 0.5 — severe deformation, no fragmentation")
            return []
        n_actual = max(1, int(n_fragments * damage_ratio))
    else:
        print(f"\n❌ FRACTURE REGIME — Full fracture ({n_fragments} fragments)")
        damage_ratio = 1.0
        n_actual     = n_fragments

    threshold = CONFIG['damage_thresh']
    t_peak    = physics['t_peak_n']

    N_scan = 50
    x_1d   = torch.linspace(0, 1, N_scan)
    y_1d   = torch.linspace(0, 1, N_scan)
    Xg, Yg = torch.meshgrid(x_1d, y_1d, indexing='ij')

    x_flat = Xg.reshape(-1, 1).to(device)
    y_flat = Yg.reshape(-1, 1).to(device)
    z_flat = torch.ones_like(x_flat)
    t_flat = torch.full_like(x_flat, t_peak)

    model_d.eval()
    with torch.no_grad():
        D_grid = model_d(x_flat, y_flat, z_flat, t_flat).reshape(N_scan, N_scan).cpu().numpy()

    D_physical   = D_grid * damage_ratio
    damaged_idx  = np.argwhere(D_physical > threshold)

    print(f"   Raw D range     : [{D_grid.min():.3f}, {D_grid.max():.3f}]")
    print(f"   Physical D range: [{D_physical.min():.3f}, {D_physical.max():.3f}]")
    print(f"   Points > {threshold}   : {len(damaged_idx)}")

    if len(damaged_idx) == 0:
        print("\n⚠️  No points exceed threshold after physical scaling")
        return []

    if len(damaged_idx) >= n_actual:
        step     = max(1, len(damaged_idx) // n_actual)
        selected = damaged_idx[::step][:n_actual]
    else:
        selected = damaged_idx[:n_actual]

    fragment_data = []
    v_debris = physics['debris_v_norm']
    x0, y0   = physics['x0'], physics['y0']

    for i, (xi, yi) in enumerate(selected):
        xpos = xi / (N_scan - 1)
        ypos = yi / (N_scan - 1)
        dx, dy = xpos - x0, ypos - y0
        dist   = max(np.sqrt(dx**2 + dy**2), 1e-6)

        vx0 = v_debris * (dx / dist) * 0.3 * damage_ratio
        vy0 = v_debris * (dy / dist) * 0.3 * damage_ratio
        vz0 = v_debris * 0.8 * damage_ratio

        fragment_data.append({
            'id'  : i,
            'x0'  : float(xpos),
            'y0'  : float(ypos),
            'z0'  : float(physics['z_launch_norm']),
            'vx0' : float(vx0),
            'vy0' : float(vy0),
            'vz0' : float(vz0),
            'D0'  : float(D_physical[xi, yi]),
            'mass': 0.01,
        })
        print(f"   Fragment {i+1}: pos=({xpos:.2f},{ypos:.2f}) "
              f"D_phys={D_physical[xi,yi]:.3f} v=({vx0:.3f},{vy0:.3f},{vz0:.3f})")

    print(f"\n✅ {len(fragment_data)} fragments identified")
    return fragment_data


class DebrisPINN(nn.Module):
    def __init__(self, layers, fragment_info):
        super().__init__()
        self.depth      = len(layers) - 1
        self.activation = nn.Tanh()
        self.layers     = nn.ModuleList([
            nn.Linear(layers[i], layers[i + 1]) for i in range(self.depth)
        ])
        self.frag = fragment_info
        for layer in self.layers:
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, t):
        out = t
        for i in range(self.depth - 1):
            out = self.activation(self.layers[i](out))
        out = self.layers[-1](out)
        return out[:, 0:1], out[:, 1:2], out[:, 2:3]


def loss_debris_pde(model_frag, t, physics, m_frag=0.01):
    g_norm, k_drag = get_debris_constants(physics, m_frag)
    x, y, z = model_frag(t)

    def grad(f, var):
        return torch.autograd.grad(
            f, var, grad_outputs=torch.ones_like(f), create_graph=True
        )[0]

    x_t, y_t, z_t    = grad(x, t), grad(y, t), grad(z, t)
    x_tt, y_tt, z_tt = grad(x_t, t), grad(y_t, t), grad(z_t, t)

    speed = torch.sqrt(x_t**2 + y_t**2 + z_t**2 + 1e-8)
    res_x = x_tt + k_drag * speed * x_t
    res_y = y_tt + k_drag * speed * y_t
    res_z = z_tt + k_drag * speed * z_t + g_norm

    ground_penalty = torch.mean(torch.clamp(-z, min=0.0)**2) * 100.0
    return (torch.mean(res_x**2) + torch.mean(res_y**2) +
            torch.mean(res_z**2) + ground_penalty)


def loss_debris_ic(model_frag, frag_info):
    t0 = torch.zeros(1, 1, dtype=torch.float32, device=device, requires_grad=True)
    mt = lambda val: torch.tensor([[val]], dtype=torch.float32, device=device)

    x, y, z = model_frag(t0)
    loss_pos = (x - mt(frag_info['x0']))**2 + (y - mt(frag_info['y0']))**2 + (z - mt(frag_info['z0']))**2

    def grad(f):
        return torch.autograd.grad(f, t0, grad_outputs=torch.ones_like(f), create_graph=True)[0]

    loss_vel = ((grad(x) - mt(frag_info['vx0']))**2 +
                (grad(y) - mt(frag_info['vy0']))**2 +
                (grad(z) - mt(frag_info['vz0']))**2)
    return torch.mean(loss_pos + loss_vel)


def validate_debris_with_ode(model_f, frag, physics, n_eval=200):
    """
    Solves the exact debris equations of motion using scipy.integrate.solve_ivp
    and computes relative L2 and max absolute trajectory error vs DebrisPINN.
    """
    g_norm, k_drag = get_debris_constants(physics, frag.get('mass', 0.01))
    t_eval = np.linspace(0.0, 1.0, n_eval)

    def ode_rhs(t, s):
        _, _, _, vx, vy, vz = s
        spd = np.sqrt(vx**2 + vy**2 + vz**2)
        return [vx, vy, vz,
                -k_drag * spd * vx,
                -k_drag * spd * vy,
                -k_drag * spd * vz - g_norm]

    y0 = [frag['x0'], frag['y0'], frag['z0'],
          frag['vx0'], frag['vy0'], frag['vz0']]
    sol = solve_ivp(ode_rhs, [0.0, 1.0], y0, t_eval=t_eval, rtol=1e-9, atol=1e-11)
    ode_pos = sol.y[:3]  # shape (3, n_eval)

    model_f.eval()
    t_tensor = torch.tensor(t_eval, dtype=torch.float32, device=device).reshape(-1, 1)
    with torch.no_grad():
        xp, yp, zp = model_f(t_tensor)
    pinn_pos = np.vstack([
        xp.cpu().numpy().ravel(),
        yp.cpu().numpy().ravel(),
        zp.cpu().numpy().ravel(),
    ])

    # Evaluate over airborne segment (z_ode >= 0) if ground is struck before t=1
    airborne = ode_pos[2] >= 0.0
    if np.any(airborne):
        diff = pinn_pos[:, airborne] - ode_pos[:, airborne]
        ref  = ode_pos[:, airborne]
    else:
        diff = pinn_pos - ode_pos
        ref  = ode_pos

    rel_l2_pct  = 100.0 * float(np.linalg.norm(diff) / (np.linalg.norm(ref) + 1e-12))
    max_abs_err = float(np.max(np.linalg.norm(diff, axis=0)))

    return {
        'rel_l2_err_%': rel_l2_pct,
        'max_abs_err' : max_abs_err,
        'ode_x'       : ode_pos[0].tolist(),
        'ode_y'       : ode_pos[1].tolist(),
        'ode_z'       : ode_pos[2].tolist(),
    }


def train_debris_pinn(model_d, physics, run_tag='default'):
    print("\n" + "=" * 55)
    print("  DEBRIS PINN — TRAINING (Adam + L-BFGS + ODE Check)")
    print("=" * 55)

    epochs      = physics['epochs']
    n_fragments = physics['n_fragments']
    resume      = physics['resume']
    lbfgs_steps = physics.get('lbfgs_steps', CONFIG['lbfgs_steps'])
    ckpt_dir    = CONFIG['ckpt_debris']
    log_dir     = f"{CONFIG['logs']}/debris"

    fragment_list = find_fracture_zones(model_d, physics, n_fragments)
    if len(fragment_list) == 0:
        print("\n✅ No fragmentation — skipping debris trajectory training")
        return [], [], []

    N_t   = 500
    t_arr = torch.linspace(0, 1, N_t, device=device).reshape(-1, 1).requires_grad_(True)

    all_models, all_histories = [], []

    for frag in fragment_list:
        fid = frag['id']
        print(f"\n── Fragment {fid + 1}/{len(fragment_list)} ──")

        ckpt_path = f"{ckpt_dir}/debris_frag{fid}_{run_tag}.pth"
        log_path  = f"{log_dir}/loss_frag{fid}_{run_tag}.json"

        model_f = DebrisPINN(CONFIG['layers_debris'], frag).to(device)
        start_epoch = 0

        if resume and os.path.exists(ckpt_path):
            ckpt = safe_torch_load(ckpt_path, map_location=device)
            model_f.load_state_dict(ckpt['model_state'])
            start_epoch = ckpt['epoch']
            print(f"  ✅ Resumed from epoch {start_epoch}")

        optimizer = torch.optim.Adam(model_f.parameters(), lr=CONFIG['lr'])
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer, step_size=CONFIG['lr_decay_step'], gamma=CONFIG['lr_decay_rate']
        )

        history   = {'total': [], 'pde': [], 'ic': [], 'epochs': []}
        best_loss = float('inf')
        save_every = max(epochs // 5, 200)

        # ── Stage 1: Adam ─────────────────────────────────
        for epoch in tqdm(range(start_epoch, start_epoch + epochs),
                          desc=f"Fragment {fid + 1} (Adam)", leave=False):
            optimizer.zero_grad()
            l_pde = loss_debris_pde(model_f, t_arr, physics, frag['mass'])
            l_ic  = loss_debris_ic(model_f, frag)
            loss  = CONFIG['w_pde'] * l_pde + CONFIG['w_ic'] * l_ic
            loss.backward()
            optimizer.step()
            scheduler.step()

            history['total'].append(loss.item())
            history['pde'].append(l_pde.item())
            history['ic'].append(l_ic.item())
            history['epochs'].append(epoch)

            if loss.item() < best_loss or (epoch % save_every == 0 and epoch > start_epoch):
                best_loss = min(loss.item(), best_loss)
                torch.save({'epoch': epoch, 'model_state': model_f.state_dict(),
                            'frag_info': frag}, ckpt_path)

        # ── Stage 2: L-BFGS Polishing ─────────────────────
        if lbfgs_steps > 0:
            lbfgs = torch.optim.LBFGS(
                model_f.parameters(), lr=1.0, max_iter=lbfgs_steps,
                history_size=50, line_search_fn='strong_wolfe'
            )
            step_counter = [start_epoch + epochs]

            def closure():
                lbfgs.zero_grad()
                l_pde = loss_debris_pde(model_f, t_arr, physics, frag['mass'])
                l_ic  = loss_debris_ic(model_f, frag)
                loss  = CONFIG['w_pde'] * l_pde + CONFIG['w_ic'] * l_ic
                loss.backward()

                history['total'].append(loss.item())
                history['pde'].append(l_pde.item())
                history['ic'].append(l_ic.item())
                history['epochs'].append(step_counter[0])
                step_counter[0] += 1
                return loss

            lbfgs.step(closure)

        # ── Stage 3: Exact ODE Validation (solve_ivp) ─────
        val = validate_debris_with_ode(model_f, frag, physics)
        frag['ode_rel_l2_%'] = val['rel_l2_err_%']
        frag['ode_max_err']  = val['max_abs_err']
        frag['ode_traj']     = (val['ode_x'], val['ode_y'], val['ode_z'])

        torch.save({
            'epoch': history['epochs'][-1],
            'model_state': model_f.state_dict(),
            'frag_info': {k: v for k, v in frag.items() if k != 'ode_traj'},
        }, ckpt_path)

        with open(log_path, 'w') as f:
            json.dump(history, f)

        print(f"  ✅ Frag {fid + 1} | Final Loss: {history['total'][-1]:.2e} | "
              f"vs solve_ivp Rel L2 Error: {val['rel_l2_err_%']:.4f}% "
              f"(Max |dr|: {val['max_abs_err']:.2e})")

        all_models.append(model_f)
        all_histories.append(history)

    print(f"\n✅ {len(fragment_list)} fragment trajectories trained & ODE-validated")
    return all_models, all_histories, fragment_list


In [ ]:
# ============================================================
# RESULTS DASHBOARD (7 Plots + Symmetry Metrics + ODE Comparison)
# ============================================================

from datetime import datetime
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def _finish_plot(fig, path):
    plt.savefig(path, dpi=150, bbox_inches='tight')
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close(fig)
    print(f"✅ Saved: {path}")


def eval_on_grid(model, t_val, N=40, mode='elastic'):
    x_1d = torch.linspace(0, 1, N)
    y_1d = torch.linspace(0, 1, N)
    Xg, Yg = torch.meshgrid(x_1d, y_1d, indexing='ij')

    x_f = Xg.reshape(-1, 1).to(device)
    y_f = Yg.reshape(-1, 1).to(device)
    z_f = torch.ones_like(x_f)
    t_f = torch.full_like(x_f, t_val)

    with torch.no_grad():
        if mode == 'elastic':
            u, v, w = model(x_f, y_f, z_f, t_f)
            disp_mag = torch.sqrt(u**2 + v**2 + w**2)
            return (Xg.numpy(), Yg.numpy(),
                    u.reshape(N, N).cpu().numpy(),
                    v.reshape(N, N).cpu().numpy(),
                    w.reshape(N, N).cpu().numpy(),
                    disp_mag.reshape(N, N).cpu().numpy())
        elif mode == 'damage':
            D = model(x_f, y_f, z_f, t_f)
            return Xg.numpy(), Yg.numpy(), D.reshape(N, N).cpu().numpy()


def eval_vm_stress(model_e, t_val, physics, N=30):
    x_1d = torch.linspace(0.05, 0.95, N)
    y_1d = torch.linspace(0.05, 0.95, N)
    Xg, Yg = torch.meshgrid(x_1d, y_1d, indexing='ij')

    x_f = Xg.reshape(-1, 1).to(device).requires_grad_(True)
    y_f = Yg.reshape(-1, 1).to(device).requires_grad_(True)
    z_f = torch.ones_like(x_f).requires_grad_(True)
    t_f = torch.full_like(x_f, t_val).requires_grad_(True)

    vm = von_mises_stress(model_e, x_f, y_f, z_f, t_f)
    return Xg.numpy(), Yg.numpy(), vm.reshape(N, N).detach().cpu().numpy()


def compute_asymmetry_error(model_e, model_d, t_val=0.3, N=40):
    """
    Quantifies emergent physical symmetry on the full domain (center impact):
      w, D even in x and y | u odd in x, even in y | v even in x, odd in y.
    """
    model_e.eval()
    model_d.eval()
    _, _, u, v, w, _ = eval_on_grid(model_e, t_val, N=N, mode='elastic')
    _, _, D          = eval_on_grid(model_d, t_val, N=N, mode='damage')

    def rel_err(a, b):
        return 100.0 * float(np.linalg.norm(a - b) / (np.linalg.norm(a) + 1e-12))

    metrics = {
        'u_x_antisym_%': rel_err(u, -u[::-1, :]),
        'u_y_sym_%'    : rel_err(u,  u[:, ::-1]),
        'v_x_sym_%'    : rel_err(v,  v[::-1, :]),
        'v_y_antisym_%': rel_err(v, -v[:, ::-1]),
        'w_x_sym_%'    : rel_err(w,  w[::-1, :]),
        'w_y_sym_%'    : rel_err(w,  w[:, ::-1]),
        'D_x_sym_%'    : rel_err(D,  D[::-1, :]),
        'D_y_sym_%'    : rel_err(D,  D[:, ::-1]),
    }
    print(f"\n📐 Emergent Symmetry Errors at t={t_val} (0% = perfectly symmetric):")
    for k, val in metrics.items():
        print(f"   {k:<16}: {val:.2f}%")
    return metrics


def plot_loss_curves(hist_e, hist_d, hist_r, run_tag, physics):
    num_plots = 2 + (1 if hist_r else 0)
    fig, axes = plt.subplots(1, num_plots, figsize=(6 * num_plots, 5))
    fig.suptitle('Training Loss History — All PINNs (Adam + L-BFGS)',
                 fontsize=14, fontweight='bold')
    if num_plots == 1:
        axes = [axes]

    configs = [(hist_e, 'Elasticity PINN'), (hist_d, 'Damage PINN')]
    if hist_r:
        configs.append((hist_r[0], 'Debris PINN (Fragment 1)'))

    for i, (hist, title) in enumerate(configs):
        ax = axes[i]
        ax.semilogy(hist['total'], label='Total', linewidth=2, color='black')
        ax.semilogy(hist['pde'],   label='PDE',   linewidth=1.5, color='royalblue')
        if 'bc' in hist:
            ax.semilogy(hist['bc'], label='BC', linewidth=1.5, color='tomato')
        if 'ic' in hist:
            ax.semilogy(hist['ic'], label='IC', linewidth=1.5, color='seagreen')
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Iteration (Adam + L-BFGS)')
        ax.set_ylabel('Loss (log scale)')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    _finish_plot(fig, f"{CONFIG['res_dashboard']}/loss_curves_{run_tag}.png")


def plot_displacement(model_e, run_tag, physics):
    t_vals = [0.1, 0.3, 0.7]
    labels = ['t = 0.1 (early)', 't = 0.3 (peak)', 't = 0.7 (late)']
    comp_labels = ['u (X-displacement)', 'v (Y-displacement)', 'w (Z-displacement)']
    cmaps = ['RdBu_r', 'RdBu_r', 'plasma']

    fig, axes = plt.subplots(3, 3, figsize=(16, 14))
    fig.suptitle('Displacement Field — Top Face (z=1)', fontsize=14, fontweight='bold')
    model_e.eval()

    for col, (tv, tl) in enumerate(zip(t_vals, labels)):
        Xg, Yg, u, v, w, _ = eval_on_grid(model_e, tv, N=40, mode='elastic')
        for row, fld in enumerate([u, v, w]):
            ax = axes[row, col]
            im = ax.contourf(Xg, Yg, fld, levels=40, cmap=cmaps[row])
            plt.colorbar(im, ax=ax, fraction=0.046)
            ax.plot(physics['x0'], physics['y0'], 'w*', markersize=12)
            ax.set_xlabel('x')
            ax.set_ylabel(comp_labels[row] + '\ny' if col == 0 else 'y', fontsize=9)
            if row == 0:
                ax.set_title(tl, fontsize=10)

    plt.tight_layout()
    _finish_plot(fig, f"{CONFIG['res_stress']}/displacement_{run_tag}.png")


def plot_von_mises(model_e, run_tag, physics):
    t_vals = [0.1, 0.3, 0.7]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Von Mises Stress — Top Face', fontsize=14, fontweight='bold')
    sigma_y = physics['sigma_y_norm']
    model_e.eval()

    for ax, tv in zip(axes, t_vals):
        Xg, Yg, vm = eval_vm_stress(model_e, tv, physics, N=30)
        im = ax.contourf(Xg, Yg, vm, levels=40, cmap='hot_r')
        plt.colorbar(im, ax=ax, fraction=0.046, label='sigma_vm (norm)')
        if vm.min() < sigma_y < vm.max():
            ax.contour(Xg, Yg, vm, levels=[sigma_y], colors='cyan', linewidths=2, linestyles='--')
        ax.plot(physics['x0'], physics['y0'], 'b*', markersize=14)
        ax.set_title(f't = {tv}', fontsize=11)
        ax.set_xlabel('x'); ax.set_ylabel('y')

    cyan_line = mpatches.Patch(color='cyan', label=f'Yield stress sigma_y={sigma_y:.4f}')
    axes[0].legend(handles=[cyan_line], fontsize=9, loc='upper left')
    plt.tight_layout()
    _finish_plot(fig, f"{CONFIG['res_stress']}/von_mises_{run_tag}.png")


def plot_damage(model_d, run_tag, physics):
    t_vals = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Damage Variable D(x,y,t) — Top Face\nD=0 (intact) -> D=1 (fractured)',
                 fontsize=13, fontweight='bold')
    axes = axes.ravel()
    threshold = CONFIG['damage_thresh']
    model_d.eval()

    for ax, tv in zip(axes, t_vals):
        Xg, Yg, D = eval_on_grid(model_d, tv, N=40, mode='damage')
        im = ax.contourf(Xg, Yg, D, levels=np.linspace(0, 1, 41),
                         cmap='RdYlBu_r', vmin=0, vmax=1)
        plt.colorbar(im, ax=ax, fraction=0.046)
        if D.min() < threshold < D.max():
            ax.contour(Xg, Yg, D, levels=[threshold], colors='black', linewidths=2)
        ax.plot(physics['x0'], physics['y0'], 'k*', markersize=12)
        dmg_pct = 100.0 * np.mean(D > threshold)
        ax.set_title(f"t = {tv}\nDamaged area: {dmg_pct:.1f}%", fontsize=10)
        ax.set_xlabel('x'); ax.set_ylabel('y')

    plt.tight_layout()
    _finish_plot(fig, f"{CONFIG['res_damage']}/damage_evolution_{run_tag}.png")


def plot_debris_3d(debris_models, fragment_list, run_tag, physics):
    fig = plt.figure(figsize=(14, 10))
    ax  = fig.add_subplot(111, projection='3d')
    colors = ['red', 'blue', 'green', 'orange', 'purple',
              'brown', 'pink', 'gray', 'cyan', 'magenta']
    t_range = torch.linspace(0, 1, 200, device=device).reshape(-1, 1)

    Xg_p, Yg_p = np.meshgrid(np.linspace(-0.5, 1.5, 10), np.linspace(-0.5, 1.5, 10))
    ax.plot_surface(Xg_p, Yg_p, np.zeros_like(Xg_p), color='lightgray', alpha=0.4)

    landing_positions = []
    for i, (model_f, frag) in enumerate(zip(debris_models, fragment_list)):
        model_f.eval()
        with torch.no_grad():
            x_t, y_t, z_t = model_f(t_range)
        x_np, y_np, z_np = x_t.cpu().numpy().ravel(), y_t.cpu().numpy().ravel(), z_t.cpu().numpy().ravel()
        c = colors[i % len(colors)]

        err_str = f" | ODE err={frag['ode_rel_l2_%']:.3f}%" if 'ode_rel_l2_%' in frag else ""
        ax.plot(x_np, y_np, z_np, color=c, linewidth=2,
                label=f'Frag {i+1} (D={frag["D0"]:.2f}{err_str})')

        # Overlay exact solve_ivp trajectory as dashed black reference
        if 'ode_traj' in frag:
            ox, oy, oz = frag['ode_traj']
            ax.plot(ox, oy, oz, color='black', linestyle='--', linewidth=1.0, alpha=0.6,
                    label='Exact ODE (solve_ivp)' if i == 0 else None)

        ax.scatter(frag['x0'], frag['y0'], frag['z0'], color=c, s=100, marker='^', zorder=5)
        land_idx = np.where(z_np <= 0.0)[0][0] if np.any(z_np <= 0.0) else -1
        ax.scatter(x_np[land_idx], y_np[land_idx], 0, color=c, s=150, marker='x', linewidths=3)

        landing_positions.append({
            'fragment'    : i + 1,
            'land_x'      : float(x_np[land_idx]),
            'land_y'      : float(y_np[land_idx]),
            'max_z'       : float(z_np.max()),
            'D0'          : frag['D0'],
            'ode_rel_l2_%': frag.get('ode_rel_l2_%', float('nan')),
        })

    plate_x, plate_y = [0, 1, 1, 0, 0], [0, 0, 1, 1, 0]
    pb = physics['plate_height_m'] / physics['L']
    ax.plot(plate_x, plate_y, [pb] * 5, 'k-', linewidth=2, label='Plate bottom')
    ax.plot(plate_x, plate_y, [physics['z_launch_norm']] * 5, 'k--', linewidth=1, alpha=0.4)
    ax.scatter(physics['x0'], physics['y0'], physics['z_launch_norm'],
               color='black', s=300, marker='*', label='Impact point', zorder=10)

    ax.set_xlabel('X (norm)'); ax.set_ylabel('Y (norm)'); ax.set_zlabel('Z (norm)')
    ax.set_title('3D Debris Trajectories (Solid = PINN, Dashed = Exact ODE)',
                 fontsize=12, fontweight='bold')
    ax.legend(loc='upper left', fontsize=8)
    ax.set_zlim(bottom=0.0)

    plt.tight_layout()
    _finish_plot(fig, f"{CONFIG['res_debris']}/trajectories_3d_{run_tag}.png")
    return landing_positions


def plot_landing_map(debris_models, fragment_list, run_tag, physics):
    fig, ax = plt.subplots(1, 1, figsize=(9, 9))
    colors = ['red', 'blue', 'green', 'orange', 'purple',
              'brown', 'pink', 'gray', 'cyan', 'magenta']
    t_range = torch.linspace(0, 1, 200, device=device).reshape(-1, 1)

    ax.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, edgecolor='black',
                               linewidth=3, label='Plate boundary'))

    for i, (model_f, frag) in enumerate(zip(debris_models, fragment_list)):
        model_f.eval()
        with torch.no_grad():
            x_t, y_t, z_t = model_f(t_range)
        x_np, y_np, z_np = x_t.cpu().numpy().ravel(), y_t.cpu().numpy().ravel(), z_t.cpu().numpy().ravel()
        c = colors[i % len(colors)]
        land_idx = np.where(z_np <= 0.0)[0][0] if np.any(z_np <= 0.0) else -1

        ax.plot(x_np, y_np, color=c, linewidth=1.5, alpha=0.6, linestyle='--')
        ax.scatter(frag['x0'], frag['y0'], color=c, s=120, marker='^', zorder=5)
        delta = np.sqrt((x_np[land_idx] - frag['x0'])**2 + (y_np[land_idx] - frag['y0'])**2)
        ax.scatter(x_np[land_idx], y_np[land_idx], color=c, s=200, marker='x',
                   linewidths=3, zorder=5, label=f'Frag {i+1} (delta={delta:.2f})')

    ax.scatter(physics['x0'], physics['y0'], color='black', s=400, marker='*',
               zorder=10, label='Impact point')
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('X (normalized)'); ax.set_ylabel('Y (normalized)')
    ax.set_title('Fragment Landing Map — Top View', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

    plt.tight_layout()
    _finish_plot(fig, f"{CONFIG['res_debris']}/landing_map_{run_tag}.png")


def plot_summary_table(physics, landing_positions, hist_e, hist_d, asym_metrics, run_tag):
    fig, ax = plt.subplots(figsize=(15, 9))
    ax.axis('off')
    fig.suptitle('Analysis Summary Report', fontsize=16, fontweight='bold', y=0.98)

    input_data = [
        ['Parameter', 'Value', 'Unit'],
        ['Projectile mass',       f"{physics.get('mass', 'N/A')}",              'kg'],
        ['Projectile velocity',   f"{physics.get('velocity', 'N/A')}",          'm/s'],
        ['Kinetic energy',        f"{physics['KE']/1e3:.2f}",                   'kJ'],
        ['Impact force',          f"{physics['F_impact']/1e6:.4f}",             'MN'],
        ['Impact pressure',       f"{physics['impact_pressure']/1e6:.2f}",      'MPa'],
        ['Plate dimensions',      f"{physics['L']}x{physics['W']}x{physics['H']}", 'm'],
        ['Plate height (bottom)', f"{physics['plate_height_m']:.3f}",           'm'],
        ['Impact location',       f"({physics['x0']:.2f}, {physics['y0']:.2f})", 'norm'],
        ['P-wave speed',          f"{physics['c_p']:.1f}",                      'm/s'],
        ['Wave cross time',       f"{physics['t_cross']*1e6:.2f}",              'us'],
    ]

    results_data = [
        ['Metric', 'Value'],
        ['Elasticity PINN final loss', f"{hist_e['total'][-1]:.6f}"],
        ['Damage PINN final loss',     f"{hist_d['total'][-1]:.6f}"],
        ['w X/Y Symmetry Error',       f"{asym_metrics['w_x_sym_%']:.2f}% / {asym_metrics['w_y_sym_%']:.2f}%"],
        ['D X/Y Symmetry Error',       f"{asym_metrics['D_x_sym_%']:.2f}% / {asym_metrics['D_y_sym_%']:.2f}%"],
        ['Fragments tracked',          str(len(landing_positions))],
    ]
    for lp in landing_positions:
        results_data.append([
            f"Frag {lp['fragment']} Landing / ODE Err",
            f"({lp['land_x']:.2f},{lp['land_y']:.2f}) | {lp['ode_rel_l2_%']:.3f}%"
        ])

    for data, bbox in [(input_data, [0.0, 0.30, 0.47, 0.62]),
                       (results_data, [0.51, 0.30, 0.49, 0.62])]:
        tbl = ax.table(cellText=data[1:], colLabels=data[0],
                       cellLoc='center', loc='upper left', bbox=bbox)
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(9)
        for (r, _), cell in tbl.get_celld().items():
            if r == 0:
                cell.set_facecolor('#2c3e50')
                cell.set_text_props(color='white', fontweight='bold')
            elif r % 2 == 0:
                cell.set_facecolor('#ecf0f1')

    ax.text(0.5, 0.22,
            f"Run tag: {run_tag}  |  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            transform=ax.transAxes, ha='center', fontsize=10, color='gray')

    plt.tight_layout()
    _finish_plot(fig, f"{CONFIG['res_dashboard']}/summary_{run_tag}.png")


def generate_dashboard(model_e, model_d, debris_models, fragment_list,
                       hist_e, hist_d, hist_r, physics, run_tag):
    print("\n" + "=" * 55)
    print("  RESULTS DASHBOARD — GENERATING")
    print("=" * 55)

    asym_metrics = compute_asymmetry_error(model_e, model_d, t_val=0.3)

    plot_loss_curves(hist_e, hist_d, hist_r, run_tag, physics)
    plot_displacement(model_e, run_tag, physics)
    plot_von_mises(model_e, run_tag, physics)
    plot_damage(model_d, run_tag, physics)

    if debris_models and fragment_list:
        landing = plot_debris_3d(debris_models, fragment_list, run_tag, physics)
        plot_landing_map(debris_models, fragment_list, run_tag, physics)
    else:
        print("⚠️  No fragments formed — skipping debris plots.")
        landing = []

    plot_summary_table(physics, landing, hist_e, hist_d, asym_metrics, run_tag)
    print(f"\n✅ Dashboard complete -> {CONFIG['res_dashboard']}")
    return asym_metrics, landing


In [ ]:
# ============================================================
# MASTER PIPELINE (Run, Load, and List Saved Runs)
# ============================================================

import os
import json
import time


def run_full_pipeline(params, status_bar=None):
    run_tag = params['timestamp']
    start   = time.time()

    print("\n" + "=" * 55)
    print("  PINN IMPACT DAMAGE ANALYZER — STARTING")
    print(f"  v={params['velocity']} m/s | m={params['mass']} kg | "
          f"epochs={params['epochs']} | N_col={params['n_col']}")
    print("=" * 55)

    physics = compute_physics_params(params)

    model_e, hist_e = train_elasticity_pinn(physics, run_tag=run_tag)
    model_d, hist_d = train_damage_pinn(model_e, physics, run_tag=run_tag)
    debris_models, hist_r, fragment_list = train_debris_pinn(model_d, physics, run_tag=run_tag)

    asym_metrics, landing = generate_dashboard(
        model_e, model_d, debris_models, fragment_list,
        hist_e, hist_d, hist_r, physics, run_tag
    )

    total = time.time() - start
    h, m, s = int(total // 3600), int((total % 3600) // 60), int(total % 60)

    print("\n" + "=" * 55)
    print(f"  ✅ PIPELINE COMPLETE in {h:02d}h {m:02d}m {s:02d}s | Run: {run_tag}")
    print("=" * 55)

    if status_bar is not None:
        status_bar.value = (
            f"<p style='color:green;font-size:13px'>"
            f"✅ Pipeline complete — {h:02d}h {m:02d}m {s:02d}s | Run: {run_tag}</p>"
        )

    return {
        'model_e'       : model_e,
        'model_d'       : model_d,
        'debris_models' : debris_models,
        'fragment_list' : fragment_list,
        'hist_e'        : hist_e,
        'hist_d'        : hist_d,
        'hist_r'        : hist_r,
        'physics'       : physics,
        'asym_metrics'  : asym_metrics,
        'landing'       : landing,
        'run_tag'       : run_tag,
    }


def load_and_display(run_tag):
    print(f"\n📂 Loading saved run: {run_tag}")
    ckpt_e = f"{CONFIG['ckpt_elastic']}/elasticity_{run_tag}.pth"
    ckpt_d = f"{CONFIG['ckpt_damage']}/damage_{run_tag}.pth"
    log_e  = f"{CONFIG['logs']}/elasticity/loss_{run_tag}.json"
    log_d  = f"{CONFIG['logs']}/damage/loss_{run_tag}.json"

    for path in [ckpt_e, ckpt_d, log_e, log_d]:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing required file: {path}")

    ce = safe_torch_load(ckpt_e, map_location=device)
    model_e = ElasticityPINN(CONFIG['layers_elast']).to(device)
    model_e.load_state_dict(ce['model_state'])
    model_e.eval()
    physics = ce['physics']

    cd = safe_torch_load(ckpt_d, map_location=device)
    model_d = DamagePINN(CONFIG['layers_damage']).to(device)
    model_d.load_state_dict(cd['model_state'])
    model_d.eval()

    debris_models, fragment_list, hist_r = [], [], []
    for i in range(CONFIG['n_fragments']):
        ckpt_f = f"{CONFIG['ckpt_debris']}/debris_frag{i}_{run_tag}.pth"
        log_f  = f"{CONFIG['logs']}/debris/loss_frag{i}_{run_tag}.json"
        if os.path.exists(ckpt_f):
            cf = safe_torch_load(ckpt_f, map_location=device)
            frag = cf['frag_info']
            mf = DebrisPINN(CONFIG['layers_debris'], frag).to(device)
            mf.load_state_dict(cf['model_state'])
            mf.eval()
            val = validate_debris_with_ode(mf, frag, physics)
            frag['ode_rel_l2_%'] = val['rel_l2_err_%']
            frag['ode_max_err']  = val['max_abs_err']
            frag['ode_traj']     = (val['ode_x'], val['ode_y'], val['ode_z'])
            debris_models.append(mf)
            fragment_list.append(frag)
            if os.path.exists(log_f):
                with open(log_f) as f:
                    hist_r.append(json.load(f))

    with open(log_e) as f:
        hist_e = json.load(f)
    with open(log_d) as f:
        hist_d = json.load(f)

    generate_dashboard(
        model_e, model_d, debris_models, fragment_list,
        hist_e, hist_d, hist_r, physics, run_tag + '_loaded'
    )


def list_saved_runs():
    ckpt_dir = CONFIG['ckpt_elastic']
    files = sorted(f for f in os.listdir(ckpt_dir) if f.endswith('.pth'))
    if not files:
        print("No saved runs found yet.")
        return []
    tags = [f.replace('elasticity_', '').replace('.pth', '') for f in files]
    for idx, tag in enumerate(tags, 1):
        print(f"  {idx}. {tag}")
    return tags


In [ ]:

# ============================================================
# KAGGLE PARAMETRIC STUDY RUNNER (18 OFAT Cases)
# ============================================================
import csv
import gc
import time
import traceback
import numpy as np
import torch
import matplotlib.pyplot as plt

# Set QUICK_TEST = True for a fast 2-minute pipeline verification
QUICK_TEST = False

BASE_PARAMS = {
    'mass'           : 1000.0,
    'velocity'       : 200.0,
    'diameter'       : 0.3,
    'angle_deg'      : 90.0,
    'length'         : 2.0,
    'width'          : 2.0,
    'thickness'      : 0.1,
    'plate_height_m' : 3.0,
    'impact_x'       : 0.5,
    'impact_y'       : 0.5,
    'epochs'         : 30   if QUICK_TEST else 2000,
    'n_col'          : 500  if QUICK_TEST else 10000,
    'lbfgs_steps'    : 10   if QUICK_TEST else 300,
    'resample_every' : 50,
    'resume'         : False,
}

# One-Factor-At-a-Time (OFAT) Parametric Matrix
EXPERIMENTS = [
    # 0. Shared Baseline (v=200, m=1000, d=0.3, angle=90, L=W=2.0, center)
    {'sweep': 'baseline',  'label': '200 m/s (Base)', 'val': 200.0, 'tag': 'S0_baseline',       'overrides': {}},

    # 1. Velocity Sweep (all other parameters fixed at baseline)
    {'sweep': 'velocity',  'label': '10',             'val': 10.0,  'tag': 'S1_vel_010',        'overrides': {'velocity': 10.0}},
    {'sweep': 'velocity',  'label': '50',             'val': 50.0,  'tag': 'S1_vel_050',        'overrides': {'velocity': 50.0}},
    {'sweep': 'velocity',  'label': '85',             'val': 85.0,  'tag': 'S1_vel_085',        'overrides': {'velocity': 85.0}},
    {'sweep': 'velocity',  'label': '100',            'val': 100.0, 'tag': 'S1_vel_100',        'overrides': {'velocity': 100.0}},
    {'sweep': 'velocity',  'label': '150',            'val': 150.0, 'tag': 'S1_vel_150',        'overrides': {'velocity': 150.0}},

    # 2. Mass Sweep (velocity FIXED at 200 m/s)
    {'sweep': 'mass',      'label': '100',            'val': 100.0,  'tag': 'S2_mass_0100',     'overrides': {'mass': 100.0}},
    {'sweep': 'mass',      'label': '180',            'val': 180.0,  'tag': 'S2_mass_0180',     'overrides': {'mass': 180.0}},
    {'sweep': 'mass',      'label': '500',            'val': 500.0,  'tag': 'S2_mass_0500',     'overrides': {'mass': 500.0}},
    {'sweep': 'mass',      'label': '2000',           'val': 2000.0, 'tag': 'S2_mass_2000',     'overrides': {'mass': 2000.0}},

    # 3. Diameter Sweep (velocity FIXED at 200 m/s, mass FIXED at 1000 kg -> constant KE)
    {'sweep': 'diameter',  'label': '0.20',           'val': 0.20,  'tag': 'S3_diam_020',       'overrides': {'diameter': 0.20}},
    {'sweep': 'diameter',  'label': '0.50',           'val': 0.50,  'tag': 'S3_diam_050',       'overrides': {'diameter': 0.50}},
    {'sweep': 'diameter',  'label': '0.60',           'val': 0.60,  'tag': 'S3_diam_060',       'overrides': {'diameter': 0.60}},

    # 4. Impact Angle Sweep (velocity FIXED at 200 m/s)
    {'sweep': 'angle',     'label': '30',             'val': 30.0,  'tag': 'S4_ang_030',        'overrides': {'angle_deg': 30.0}},
    {'sweep': 'angle',     'label': '60',             'val': 60.0,  'tag': 'S4_ang_060',        'overrides': {'angle_deg': 60.0}},

    # 5. Impact Location Sweep (velocity FIXED at 200 m/s)
    {'sweep': 'location',  'label': '(0.25,0.50)',    'val': 1.0,   'tag': 'S5_loc_edge',       'overrides': {'impact_x': 0.25, 'impact_y': 0.50}},
    {'sweep': 'location',  'label': '(0.25,0.25)',    'val': 2.0,   'tag': 'S5_loc_corner',     'overrides': {'impact_x': 0.25, 'impact_y': 0.25}},

    # 6. Plate Span Sweep (velocity FIXED at 200 m/s)
    {'sweep': 'plate_span','label': '5x5 m',          'val': 5.0,   'tag': 'S6_span_5x5',       'overrides': {'length': 5.0, 'width': 5.0}},
]

csv_path = f"{CONFIG['exports']}/parametric_summary.csv"
CSV_FIELDS = [
    'tag', 'sweep', 'label', 'velocity', 'mass', 'diameter', 'angle_deg',
    'length', 'impact_x', 'impact_y', 'impact_pressure_MPa', 'KE_kJ',
    'regime', 'elast_loss', 'damage_loss', 'max_abs_w_t03', 'peak_vm_t03',
    'damaged_area_pct_t10', 'w_sym_err_pct', 'D_sym_err_pct',
    'n_fragments', 'mean_ode_err_pct', 'runtime_sec'
]

# Suppress inline rendering of ~126 individual plots during the loop (still saved as high-res PNGs)
_orig_in_notebook = IN_NOTEBOOK
IN_NOTEBOOK = False

records = []
with open(csv_path, 'w', newline='') as f:
    csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()

batch_start = time.time()
print(f"🚀 Starting Parametric Study: {len(EXPERIMENTS)} cases")

for idx, exp in enumerate(EXPERIMENTS, 1):
    print("\n" + "#" * 65)
    print(f"  CASE {idx}/{len(EXPERIMENTS)} : {exp['tag']}  (sweep={exp['sweep']})")
    print("#" * 65)

    p = dict(BASE_PARAMS)
    p.update(exp['overrides'])
    p['timestamp'] = exp['tag']

    t0 = time.time()
    try:
        res = run_full_pipeline(p)
        phys = res['physics']

        # Extract quantitative physical response metrics on top face (z=1)
        res['model_e'].eval()
        res['model_d'].eval()
        _, _, _, _, w_t03, _ = eval_on_grid(res['model_e'], 0.3, N=40, mode='elastic')
        _, _, vm_t03         = eval_vm_stress(res['model_e'], 0.3, phys, N=30)
        _, _, D_t10          = eval_on_grid(res['model_d'], 1.0, N=40, mode='damage')

        p_mpa = phys['impact_pressure'] / 1e6
        if p_mpa < CONFIG['sigma_y'] / 1e6:
            regime = 'Elastic'
        elif p_mpa < CONFIG['sigma_u'] / 1e6:
            regime = 'Plastic'
        else:
            regime = 'Fracture'

        asym = res['asym_metrics']
        ode_errs = [lp['ode_rel_l2_%'] for lp in res['landing']
                    if not np.isnan(lp.get('ode_rel_l2_%', np.nan))]
        mean_ode = float(np.mean(ode_errs)) if ode_errs else 0.0

        row = {
            'tag'                 : exp['tag'],
            'sweep'               : exp['sweep'],
            'label'               : exp['label'],
            'velocity'            : p['velocity'],
            'mass'                : p['mass'],
            'diameter'            : p['diameter'],
            'angle_deg'           : p['angle_deg'],
            'length'              : p['length'],
            'impact_x'            : p['impact_x'],
            'impact_y'            : p['impact_y'],
            'impact_pressure_MPa' : round(p_mpa, 2),
            'KE_kJ'               : round(phys['KE'] / 1e3, 2),
            'regime'              : regime,
            'elast_loss'          : float(res['hist_e']['total'][-1]),
            'damage_loss'         : float(res['hist_d']['total'][-1]),
            'max_abs_w_t03'       : float(np.max(np.abs(w_t03))),
            'peak_vm_t03'         : float(np.max(vm_t03)),
            'damaged_area_pct_t10': float(100.0 * np.mean(D_t10 > CONFIG['damage_thresh'])),
            'w_sym_err_pct'       : float(0.5 * (asym['w_x_sym_%'] + asym['w_y_sym_%'])),
            'D_sym_err_pct'       : float(0.5 * (asym['D_x_sym_%'] + asym['D_y_sym_%'])),
            'n_fragments'         : len(res['fragment_list']),
            'mean_ode_err_pct'    : round(mean_ode, 4),
            'runtime_sec'         : round(time.time() - t0, 1),
        }
        records.append(row)

        with open(csv_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row)

        print(f"📊 Recorded {exp['tag']}: P={row['impact_pressure_MPa']} MPa ({regime}) | "
              f"max|w|={row['max_abs_w_t03']:.5f} | D_area={row['damaged_area_pct_t10']:.1f}% | "
              f"w_sym={row['w_sym_err_pct']:.2f}% | Frags={row['n_fragments']}")

    except Exception as e:
        print(f"❌ Case {exp['tag']} failed: {e}")
        traceback.print_exc()

    # Free figures and GPU memory between runs
    plt.close('all')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

IN_NOTEBOOK = _orig_in_notebook

total_min = (time.time() - batch_start) / 60.0
print("\n" + "=" * 65)
print(f"✅ PARAMETRIC STUDY COMPLETE: {len(records)}/{len(EXPERIMENTS)} runs in {total_min:.1f} min")
print(f"📄 Summary CSV saved -> {csv_path}")
print("=" * 65)

# ============================================================
# GENERATE 6-PANEL COMPARATIVE PARAMETRIC PLOT
# ============================================================
base_row = next((r for r in records if r['tag'] == 'S0_baseline'), None)

def get_sweep_rows(sweep_name, x_key):
    rows = [r for r in records if r['sweep'] == sweep_name]
    if base_row is not None:
        rows.append(base_row)
    return sorted(rows, key=lambda r: r[x_key])

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Parametric Sensitivity Analysis (OFAT around Baseline: v=200 m/s, m=1000 kg, d=0.3 m, 90 deg)',
             fontsize=13, fontweight='bold')

numeric_sweeps = [
    (axes[0, 0], 'velocity', 'velocity',  'Velocity (m/s)',       '1. Effect of Velocity'),
    (axes[0, 1], 'mass',     'mass',      'Projectile Mass (kg)', '2. Effect of Mass (v=200 m/s fixed)'),
    (axes[0, 2], 'diameter', 'diameter',  'Diameter (m)',         '3. Effect of Diameter (v=200, m=1000 fixed)'),
    (axes[1, 0], 'angle',    'angle_deg', 'Impact Angle (deg)',   '4. Effect of Angle (v=200 m/s fixed)'),
]

for ax, sname, xkey, xlabel, title in numeric_sweeps:
    rows = get_sweep_rows(sname, xkey)
    if not rows:
        continue
    xs   = [r[xkey] for r in rows]
    ws   = [r['max_abs_w_t03'] for r in rows]
    dmgs = [r['damaged_area_pct_t10'] for r in rows]

    l1 = ax.plot(xs, ws, 'o-', color='royalblue', linewidth=2, label='Max |w| (t=0.3)')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Max |w| (norm)', color='royalblue')
    ax.tick_params(axis='y', labelcolor='royalblue')
    ax.grid(True, alpha=0.3)
    ax.set_title(title, fontsize=11)

    ax2 = ax.twinx()
    l2 = ax2.plot(xs, dmgs, 's--', color='crimson', linewidth=2, label='Damaged Area % (t=1.0)')
    ax2.set_ylabel('Damaged Area (%)', color='crimson')
    ax2.tick_params(axis='y', labelcolor='crimson')
    ax2.set_ylim(-5, 105)

    lines = l1 + l2
    ax.legend(lines, [l.get_label() for l in lines], loc='upper left', fontsize=8)

# Panel 5: Impact Location (categorical bar chart: symmetry error & peak stress)
ax_loc = axes[1, 1]
loc_rows = ([base_row] if base_row else []) + [r for r in records if r['sweep'] == 'location']
if loc_rows:
    loc_labels = ['(0.50,0.50)\nCenter' if r['tag'] == 'S0_baseline' else r['label'] for r in loc_rows]
    x_pos = np.arange(len(loc_rows))
    ax_loc.bar(x_pos - 0.18, [r['w_sym_err_pct'] for r in loc_rows], 0.35, label='w Symmetry Error (%)', color='teal')
    ax_loc.bar(x_pos + 0.18, [r['damaged_area_pct_t10'] for r in loc_rows], 0.35, label='Damaged Area (%)', color='coral')
    ax_loc.set_xticks(x_pos)
    ax_loc.set_xticklabels(loc_labels)
    ax_loc.set_ylabel('Percentage (%)')
    ax_loc.set_title('5. Effect of Impact Location (v=200 fixed)', fontsize=11)
    ax_loc.legend(fontsize=8)
    ax_loc.grid(True, axis='y', alpha=0.3)

# Panel 6: Plate Span (2x2 m vs 5x5 m)
ax_span = axes[1, 2]
span_rows = ([base_row] if base_row else []) + [r for r in records if r['sweep'] == 'plate_span']
if span_rows:
    span_labels = ['2x2 m' if r['tag'] == 'S0_baseline' else r['label'] for r in span_rows]
    x_pos = np.arange(len(span_rows))
    ax_span.bar(x_pos - 0.18, [r['peak_vm_t03'] * 1e3 for r in span_rows], 0.35, label='Peak sigma_vm (x1e-3)', color='darkorange')
    ax_span.bar(x_pos + 0.18, [r['w_sym_err_pct'] for r in span_rows], 0.35, label='w Symmetry Error (%)', color='slateblue')
    ax_span.set_xticks(x_pos)
    ax_span.set_xticklabels(span_labels)
    ax_span.set_ylabel('Value')
    ax_span.set_title('6. Effect of Plate Span (v=200 fixed)', fontsize=11)
    ax_span.legend(fontsize=8)
    ax_span.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
comp_path = f"{CONFIG['exports']}/parametric_comparison.png"
plt.savefig(comp_path, dpi=180, bbox_inches='tight')
plt.show()
print(f"✅ Comparative plot saved -> {comp_path}")
